# Preprocessing PolEmo: 10 losowych tekstów

Plik `.txt` w formacie fastText: **jedna linia = jeden cały dokument** (opinia), a `__label__...` na końcu to oryginalna etykieta. Tekst jest tokenizowany spacjami, więc trzeba go odtworzyć do czytelnej formy.

In [4]:
import random
import re
from pathlib import Path

import pandas as pd

DATA = Path("../data/dataset_conll/all.text.train.txt")
SEED = 42

In [5]:
print(DATA)

../data/dataset_conll/all.text.train.txt


In [6]:
def detokenize(text: str) -> str:
    text = re.sub(r" ([.,!?;:)%])", r"\1", text)
    return re.sub(r"\( ", "(", text)


def parse(line: str) -> dict:
    raw, label = line.rsplit(" __label__", 1)
    return {"raw": raw.strip(), "text": detokenize(raw.strip()), "gold": label.strip()}


lines = [l for l in DATA.read_text(encoding="utf-8").splitlines() if l.strip()]
df = pd.DataFrame([{"id": i, **parse(l)} for i, l in enumerate(lines)])
print(len(df), "dokumentów")
df["gold"].value_counts()

6573 dokumentów


gold
meta_minus_m    2469
meta_plus_m     1824
meta_amb        1309
meta_zero        971
Name: count, dtype: int64

In [7]:
sample = df.sample(10, random_state=SEED)
for _, r in sample.iterrows():
    print(f"[{r.id}] ({r.gold})")
    print("RAW :", r.raw[:200])
    print("TEXT:", r.text[:200])
    print()

[611] (meta_amb)
RAW : na plus muszę zaliczyć położenie ( idealne na nocleg tranzytowy ) i śniadanie . Obsługa bardzo miła - na recepcji przywitał nas właściciel , a śniadanie podawała jego żona . Pewnym mankamentem jest br
TEXT: na plus muszę zaliczyć położenie (idealne na nocleg tranzytowy) i śniadanie. Obsługa bardzo miła - na recepcji przywitał nas właściciel, a śniadanie podawała jego żona. Pewnym mankamentem jest brak re

[2498] (meta_amb)
RAW : Cóż , w tej cenie nie ma co się spodziewać cudów , dźwięk jest , basy praktycznie zerowe co jednak można korektorem w karcie dźwiękowej podbić i wówczas brzmią nawet znośnie , choć jest czysto i w mia
TEXT: Cóż, w tej cenie nie ma co się spodziewać cudów, dźwięk jest, basy praktycznie zerowe co jednak można korektorem w karcie dźwiękowej podbić i wówczas brzmią nawet znośnie, choć jest czysto i w miarę p

[1199] (meta_amb)
RAW : Hotel 4 gwiazdkowy zbudowany chyba w latach 90 gdzie pojęcie wysokiej jakości czy luksusu odbiegało od teraźnie

In [ ]:
# format importu do Label Studio (bez etykiety oryginalnej)
[{"data": {"id": int(r.id), "text": r.text}} for _, r in sample.iterrows()][:2]

In [14]:
import json
import shutil

OUT = Path("../annotation")
shutil.rmtree(OUT, ignore_errors=True)


def save(folder: str, name: str, part: pd.DataFrame) -> None:
    d = OUT / folder
    d.mkdir(parents=True, exist_ok=True)
    tasks = [{"data": {"id": int(r.id), "text": r.text}} for _, r in part.iterrows()]
    gold = {int(r.id): r.gold for _, r in part.iterrows()}
    (d / f"{name}.json").write_text(
        json.dumps(tasks, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (d / f"gold_{name}.json").write_text(
        json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8"
    )


save("sample1", "sample1", sample)
print(sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*.json")))


['sample1/gold_sample1.json', 'sample1/sample1.json']


In [15]:
# sample2: kolejne 10 tekstów, rozłączne z sample1
rest = df[~df["id"].isin(sample["id"])]
sample2 = rest.sample(10, random_state=SEED)
save("sample2", "sample2", sample2)
rest = rest.drop(sample2.index)
print(len(rest), "dokumentów pozostało")


6553 dokumentów pozostało


In [16]:
N_PERSON = 5
used = set(sample["id"]) | set(sample2["id"])

for person in ["mikolaj", "dominik", "piotrek"]:
    part = rest.sample(N_PERSON, random_state=SEED)
    rest = rest.drop(part.index)
    assert not used & set(part["id"])
    used |= set(part["id"])
    save(f"sample_{person}", f"sample_{person}", part)

print(sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*.json")))


['sample1/gold_sample1.json', 'sample1/sample1.json', 'sample2/gold_sample2.json', 'sample2/sample2.json', 'sample_dominik/gold_sample_dominik.json', 'sample_dominik/sample_dominik.json', 'sample_mikolaj/gold_sample_mikolaj.json', 'sample_mikolaj/sample_mikolaj.json', 'sample_piotrek/gold_sample_piotrek.json', 'sample_piotrek/sample_piotrek.json']
